# run_servo_setup: assign servo IDs and calibrate

Run notebook (needs the arm + Waveshare Bus Servo Adapter). Milestone: *Arm assembled, servo
identifiers assigned, calibrated*. Two routes: LeRobot's own tools (recommended first, they
write LeRobot calibration files used by run_teleop) or our `ServoBus` (servo_bus.ipynb), whose
calibration is what the Python/C++ paths use. Keep both consistent.

**TODO**
- [ ] find the adapter port (`ls /dev/ttyACM* /dev/ttyUSB*`), check 1 Mbaud
- [ ] assign IDs one servo at a time (only ONE servo on the bus while writing its ID)
- [ ] calibrate: zero offsets so URDF zero == tick 2048, sign per joint, range of motion
- [ ] save to `gambit_bringup/config/arm_calibration.yaml`; load it in servo_bus.CALIBRATION
- [ ] repeat for the leader arm (LeRobot teleop)

In [ ]:
%run ./square_map.ipynb
%run ./servo_bus.ipynb
PORT = "/dev/ttyACM0"   # TODO: set

## Option A: LeRobot (recommended first)

```bash
pip install 'lerobot[feetech]'
lerobot-find-port
lerobot-setup-motors --robot.type=so101_follower --robot.port=/dev/ttyACM0
lerobot-calibrate    --robot.type=so101_follower --robot.port=/dev/ttyACM0 --robot.id=gambit_follower
lerobot-setup-motors --teleop.type=so101_leader --teleop.port=/dev/ttyACM1
lerobot-calibrate    --teleop.type=so101_leader --teleop.port=/dev/ttyACM1 --teleop.id=gambit_leader
```
(Command names change between LeRobot releases; check `lerobot --help` for your version.)

## Option B: our ServoBus

Assign one ID: connect only the new servo (factory ID 1), unlock EEPROM, write `Reg.ID`, lock.

In [ ]:
def assign_id(bus, new_id: int, current_id: int = 1):
    """Change one servo's ID (only that servo on the bus!)."""
    # Design: unlock EEPROM (Reg.LOCK = 0), write Reg.ID, lock again at the NEW id, then ping
    # the new id to confirm. Verify the register addresses against the datasheet first: a
    # wrong EEPROM write can brick the servo's settings.
    # TODO: implement
    raise NotImplementedError

def scan(bus, ids=range(0, 20)) -> list:
    """IDs that answer a PING."""
    # TODO: implement (1 line)
    raise NotImplementedError

def record_range(bus, seconds: float = 20) -> dict:
    """{joint: (min_ticks, max_ticks)} while the user moves every joint by hand."""
    # Design: torque off, read positions at ~50 Hz for `seconds`, track min/max per joint.
    # Then, with the arm held at the URDF zero pose, offset_ticks = 2048 - raw per joint, and
    # decide each sign by moving a joint in its URDF-positive direction.
    # TODO: implement
    raise NotImplementedError

# Step 1: with ServoBus(PORT) as bus: scan(bus) -> expect [1] for a factory-new servo.
# Step 2: assign_id(bus, n) for n = 1..6, one servo connected at a time.
# Step 3: all six connected: scan(bus) == [1..6]; record_range(bus); fill CALIBRATION.

In [ ]:
def save_calibration(cal, path=CONFIG_DIR / "arm_calibration.yaml"):
    """Write {joint: {offset_ticks, sign}} to YAML; return the path."""
    # Design: plain dict -> yaml.safe_dump; servo_bus.CALIBRATION should load the same file.
    # TODO: implement
    raise NotImplementedError

# Step 4: after calibration: save_calibration(CALIBRATION)